In [0]:
%run "../includes/configuration"


In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")


### Ingestión del archivo "movie_cast.json"

###Paso 1 - Leer el archivo JSON usando "DataFrameReader" de Spark
####La documentación para los comandos de Spark la sacamos de https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

movie_cast_schema = StructType(fields = [
    StructField("movieId", IntegerType(), True),
    StructField("personId", IntegerType(), True),
    StructField("characterName", StringType(), True),
    StructField("genderId", IntegerType(), True),
    StructField("castOrder", IntegerType(), True),
])


In [0]:
#movie_cast_df = spark.read.schema(movie_cast_schema).option("multiLine", True).json("abfss://bronze@moviee.dfs.core.windows.net/movie_cast.json")

movie_cast_df = spark.read.schema(movie_cast_schema).option("multiLine", True).json(f"{bronze_folder_path}/movie_cast.json")

In [0]:
movie_cast_df.printSchema()

In [0]:
display(movie_cast_df)

### Paso 2 - Renombrar las columnas y añadir columnas nuevas

In [0]:
from pyspark.sql.functions import current_timestamp, lit

#movie_cast_with_columns_df = movie_cast_df \
#                        .withColumnsRenamed({"movieId": "movie_id",
#                                             "personId": "person_id",
#                                             "characterName": "character_name"}) \
#                        .withColumn("ingestion_date", current_timestamp()) \
#                        .withColumn("environment", lit("production"))

movie_cast_with_columns_df = movie_cast_df \
                             .withColumnsRenamed({"movieId": "movie_id",
                                             "personId": "person_id",
                                             "characterName": "character_name"})

movie_cast_with_columns_df = add_ingestion_date(movie_cast_with_columns_df) \
                             .withColumn("environment", lit(v_environment))


In [0]:
display(movie_cast_with_columns_df)

### Paso 3 - Eliminar las columnas no deseadas del DataFrame

In [0]:
from pyspark.sql.functions import col

In [0]:
movie_cast_final_df = movie_cast_with_columns_df.drop(col("genderId"),col("castOrder"))

In [0]:
display(movie_cast_final_df)

### Paso 4 - Escribir la salida en un formato "Parquet"

In [0]:
#movie_cast_final_df.write.mode("overwrite").parquet("abfss://silver@moviee.dfs.core.windows.net/movie_casts")

movie_cast_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/movie_casts")

In [0]:
#display(spark.read.parquet("abfss://silver@moviee.dfs.core.windows.net/movie_casts"))

display(spark.read.parquet(f"{silver_folder_path}/movie_casts"))

### Paso 5 - Escribir datos en una managed table en el contenedor silver

#### Esto es un cambio posterior en el notebook, pasaría a reemplazar al paso 4, ya que ahora no quiero más generar archivos de salida en la capa silver a partir del dataframe, sino meter esa info en una tabla administrada por spark, pero dejo igual lo del paso 4, ya que esto se crea dentro de la carpeta _unitystorage y no pisa lo del paso 4

In [0]:
movie_cast_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.movies_casts")

In [0]:
%sql
SELECT * FROM movie_silver.movies_casts

In [0]:
dbutils.notebook.exit("Success")